# Quantifying ammonia in the five-hour electrolysis

This example follows the salicylate workflow from blank-corrected absorbance to ammonia concentration, endpoint recovery, Faradaic efficiency, and average production rate. The repository contains the assay signals, calibration points, water-blank spectra, and run configuration for this experiment. The example starts from the deposited assay signals.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "styles" / "manuscript.mplstyle").is_file()
     and (path / "src" / "extended_operation.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Start Jupyter from the repository or its notebooks folder.")

sys.path.insert(0, str(root))
from src.extended_operation import read_inputs, run_extended_operation

style = root / "styles" / "manuscript.mplstyle"
use_tex = False  # Set to True if the manuscript's LaTeX dependencies are installed.
config, standards, _, _, _ = read_inputs(root)
result = run_extended_operation(root)
summary = result["summary"]

## Calibration

The analysis fits low- and high-range standards by unweighted linear regression with a free intercept. Both curves are shown below. These were used for the five-hour experiment's calibration series.

In [ ]:
with plt.style.context(style):
    plt.rcParams["text.usetex"] = use_tex
    fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.25), layout="constrained")
    for ax, (name, fit) in zip(axes, summary["calibration_fits"].items()):
        x, y = zip(*fit["data"])
        fitted = [fit["intercept"] + fit["slope"] * value for value in x]
        ax.scatter(x, y)
        ax.plot(x, fitted, linestyle="--")
        ax.set(xlabel=r"Ammonia concentration ($\mu$M)",
               ylabel="Blank-corrected differential absorbance (AU)", title=name.capitalize())
        ax.set_box_aspect(1)
    plt.show()

## From assay signal to sample concentration

For a single electrolyte sample, the table shows the selected calibration, the inferred assay concentration, and the concentration after correcting for dilution.

In [ ]:
samples = result["samples"]
sample = samples.loc[samples["compartment"].eq("e"), "file"].iloc[0]
columns = ["dilution", "signal_AU", "fc_calibration", "assay_uM",
           "concentration_uM", "valid"]
display(result["readings"].loc[result["readings"]["file"].eq(sample), columns])
display(samples.loc[samples["file"].eq(sample)])

## Endpoint recovery and performance

The endpoint calculation combines final collections and withdrawn aliquots, with the initial ammonia background subtracted. The reported FE and rate use net recovery over the full five-hour electrolysis. Uncertainties describe propagated analytical variation from a single electrolysis.

In [ ]:
display(result["recovery"])
print(f"Net ammonia: {summary['recovered_net_umol']:.6f} µmol")
print(f"FE: {summary['endpoint_FE_percent']:.6f} ± {summary['endpoint_FE_SD_percent']:.6f}%")
print(f"Average rate: {summary['average_rate_nmol_s_cm2']:.6f} "
      f"± {summary['average_rate_SD_nmol_s_cm2']:.6f} nmol s⁻¹ cm⁻²")